# NBA points scoring discovery

Explore what drives `pts`, `pts_per_min`, and `minutes` using existing training parquets + optional `nba_api` probes.

Spec: `docs/superpowers/specs/2026-07-26-nba-pts-scoring-discovery-design.md`

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve repo root (directory that contains data/)
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    for cand in [ROOT.parent, *ROOT.parents]:
        if (cand / "data").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
print("cwd:", Path.cwd())

In [ ]:
from src.pipeline.features.context_features import ContextFeatureEngineer
from models.shared.scoring_discovery import (
    build_coverage_map,
    derive_pts,
    lineage_for,
    merge_driver_shortlist,
    rank_univariate,
    season_rank_stability,
    split_feature_pools,
)

SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
HOLDOUT_SEASON = "2025-26"
TARGETS = ["pts", "pts_per_min", "minutes"]
RANDOM_SEED = 42
RUN_ENDPOINT_PROBES = False  # set True only when live nba_api pulls are desired
np.random.seed(RANDOM_SEED)

In [ ]:
frames = []
missing = []
for yr in SEASONS:
    path = Path(f"data/processed/{yr}_Regular_Season_training_data.parquet")
    if not path.exists():
        missing.append(yr)
        print(f"⚠ missing parquet for {yr}: {path}")
        continue
    season_df = pd.read_parquet(path)
    season_df = ContextFeatureEngineer(league="nba", season=yr).enrich(season_df)
    frames.append(season_df)
    print(f"✓ {yr}: {len(season_df):,} rows")

if not frames:
    raise FileNotFoundError("No season parquets found under data/processed/")

df = pd.concat(frames, ignore_index=True)
df = derive_pts(df)
df = df[(df["minutes"] >= 5) | (df["starting"] == 1)].copy()
print(f"Combined: {len(df):,} rows × {df.shape[1]} cols | missing seasons: {missing or 'none'}")
df[["season_year", "pts", "pts_per_min", "minutes"]].describe()

In [ ]:
dupes = df.duplicated(subset=["game_id", "player_id"]).sum()
print(f"Duplicate game_id+player_id: {dupes}")
print(df.groupby("season_year").size())
print("Target nulls:", {t: int(df[t].isna().sum()) for t in TARGETS})

## Wishlist coverage

Status is based on columns present **after** load+enrich — not aspirational names.
`partial` = proxy only (e.g. contested FGA ≈ contest rate).

In [ ]:
coverage_df = build_coverage_map(df.columns)
display(coverage_df)
print(coverage_df["status"].value_counts())

## Scoring anatomy (same-game)

**NOT model features.** These are contemporaneous associations that explain how points are produced in the same game. Using them pre-tip is leakage.

In [ ]:
pools = split_feature_pools(df.columns, targets=TARGETS)
print({k: len(v) for k, v in pools.items()})
# leakage audit
overlap = set(pools["same_game"]) & set(pools["predictive"])
print("same_game ∩ predictive:", overlap or "∅ (ok)")

anatomy_ranks = {}
for target in TARGETS:
    ranks = rank_univariate(df, pools["same_game"], target, random_state=RANDOM_SEED)
    anatomy_ranks[target] = ranks
    print(f"\n=== same-game drivers of {target} ===")
    display(ranks.head(20))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [
    ("fga_per_min", "pts"),
    ("tchs_per_min", "pts_per_min"),
    ("minutes", "pts"),
]
sample = df.sample(min(8000, len(df)), random_state=RANDOM_SEED)
for ax, (x, y) in zip(axes, pairs):
    if x not in sample.columns or y not in sample.columns:
        ax.set_title(f"missing {x}/{y}")
        continue
    ax.scatter(sample[x], sample[y], s=4, alpha=0.15)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
plt.tight_layout()
plt.show()